In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='ISO-8859-1')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])


columns_to_drop = [
    # PII / no analytical value
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',

    # Empty or near-useless
    'Product Description',      # 100% missing
    'Product Image',            # just a URL
    'Order Zipcode',            # ~86% missing
    'Product Status',            # ~99% missing
    # Redundant duplicate IDs
    'Order Customer Id',        # duplicate of Customer Id
    'Product Category Id',      # duplicate of Category Id
    'Order Item Cardprod Id',   # duplicate of Product Card Id
]
df_clean = df.drop(columns=columns_to_drop)

df_clean = df_clean.rename(columns={'shipping date (DateOrders)': 'Shipping_Date', 
                                    'order date (DateOrders)': 'Order_Date',
                                    'Days for shipping (real)' : 'Shipping_days',
                                    'Days for shipment (scheduled)' : 'Scheduled_shipping_days', 
                                    'Order Profit Per Order' : 'Profit_per_order',
                                      'Order Item Discount Rate' : 'Discount_rate', 
                                      'Order Item Profit Ratio' : 'Profit_ratio',
                                      'Order Item Quantity' : 'Quantity',
                                      'Order Item Product Price' : 'Prroduct_price',
                                      'Order Item Discount' : 'Discount',
                                      'Order Item Total' : 'Order_item_total',
                                      'Shipping Mode' : 'Shipping_mode',
                                      'Order Region' : 'Order_region',
                                      'Order State': 'Order_state',
                                      'Order Status' : 'Order_status'})   #Renaming columns

df_clean = df_clean.fillna({'Customer Zipcode': '00000'})   #Replacing values in customer zipcode with '00000'

In [2]:
df_clean.groupby('Shipping_mode')['Late_delivery_risk'].mean().rename('late_delivery_rate').reset_index()

,Shipping_mode,late_delivery_rate
0,First Class,0.953225
1,Same Day,0.457430
2,Second Class,0.766328
3,Standard Class,0.380717


In [3]:
df_clean['Late_delivery_risk'].mean()

np.float64(0.5482913155955883)

In [4]:
df_regions = (df_clean
              .groupby('Order_region')['Late_delivery_risk']
              .mean().rename('late_delivery_rate')
              .reset_index().sort_values(by='late_delivery_rate', ascending= False)
              .reset_index(drop=True)
              .head(10)
              )

In [5]:
df_market = (df_clean
             .groupby('Market')['Late_delivery_risk']
             .median().rename('late_delivery_rate')
             .reset_index()
             )

In [6]:
df_clean['shipping_gap'] = df_clean['Shipping_days'] - df_clean['Scheduled_shipping_days']
df_clean.groupby('Shipping_mode')['shipping_gap'].mean().rename('avg_shipping_gap').reset_index()

,Shipping_mode,avg_shipping_gap
0,First Class,1.000000
1,Same Day,0.478279
2,Second Class,1.990828
3,Standard Class,-0.004093


In [7]:
late_rate = df_clean.groupby('Shipping_mode')['Late_delivery_risk'].mean().rename('late_delivery_rate')
gap = df_clean.groupby('Shipping_mode')['shipping_gap'].mean().rename('avg_shipping_gap')

comparison = pd.concat([late_rate, gap], axis=1).reset_index()


In [8]:
comparison

,Shipping_mode,late_delivery_rate,avg_shipping_gap
0,First Class,0.953225,1.000000
1,Same Day,0.457430,0.478279
2,Second Class,0.766328,1.990828
3,Standard Class,0.380717,-0.004093


In [9]:
late_rate = df_clean.groupby('Order_region')['Late_delivery_risk'].mean().rename('late_delivery_rate')
gap = df_clean.groupby('Order_region')['shipping_gap'].mean().rename('avg_shipping_gap')

comparison_1 = pd.concat([late_rate, gap], axis=1).reset_index()

In [10]:
comparison_1

,Order_region,late_delivery_rate,avg_shipping_gap
0,Canada,0.488008,0.391032
1,Caribbean,0.530777,0.546526
2,Central Africa,0.579606,0.639833
3,Central America,0.547546,0.561942
4,Central Asia,0.553345,0.645570
5,East Africa,0.559395,0.570734
6,East of USA,0.556616,0.584816
7,Eastern Asia,0.543269,0.566484
8,Eastern Europe,0.556633,0.579847
9,North Africa,0.545173,0.552290


In [11]:
late_rate = df_clean.groupby('Market')['Late_delivery_risk'].mean().rename('late_delivery_rate')
gap = df_clean.groupby('Market')['shipping_gap'].mean().rename('avg_shipping_gap')

comparison_2 = pd.concat([late_rate, gap], axis=1).reset_index()

In [12]:
comparison_2

,Market,late_delivery_rate,avg_shipping_gap
0,Africa,0.545893,0.560014
1,Europe,0.552078,0.570843
2,LATAM,0.543552,0.557836
3,Pacific Asia,0.550460,0.569365
4,USCA,0.548006,0.568859
